# Tunix-Med: Final Knowledge Evaluation

This notebook re-evaluates the **fine-tuned `google/gemma-3-1b-it` model** after training with **Tunix**. We compare its performance against the baseline to quantify the knowledge gain in the medical domain.

### Evaluation Pipeline
We use the same 25 cardiology questions as the baseline evaluation.

In [1]:
import os
import warnings
import torch
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings("ignore")


def info_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()

# Gemma 3 overflows float16; always use bfloat16 on modern GPUs.
# Recent Transformers deprecated `torch_dtype` in favour of `dtype`.
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)

BASE_MODEL   = "google/gemma-3-1b-it"
ADAPTER_PATH = "tunix-medical-model"

print(f"Device : {device}  |  dtype : {dtype}")
print("Loading base model ...")
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,          # ← current Transformers; torch_dtype is deprecated
    device_map=device,
)

print("Loading LoRA adapter ...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Merging adapter into base weights ...")
model = model.merge_and_unload()
model.eval()
print("Model ready.")


Device : cuda  |  dtype : torch.bfloat16
Loading base model ...
Loading LoRA adapter ...
Merging adapter into base weights ...
Model ready.


In [2]:
medical_qa = [
    # --- HYPERTENSION (3) ---
    {
        "Question": "What is the recommended first-line treatment for uncomplicated hypertension according to ESC guidelines?",
        "Answer": "ACE inhibitors or ARBs, combined with a calcium channel blocker or a thiazide/thiazide-like diuretic.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    {
        "Question": "What is the definition of Stage 1 hypertension according to the 2017 ACC/AHA guidelines?",
        "Answer": "Systolic blood pressure 130–139 mmHg or diastolic blood pressure 80–89 mmHg.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    {
        "Question": "What is the definition of hypertensive emergency, and what is the initial target for blood pressure reduction in the first hour?",
        "Answer": "Hypertensive emergency is severe hypertension (typically SBP > 180 mmHg) with acute end-organ damage. BP should be reduced by no more than 25% in the first hour using IV agents (e.g., labetalol, nicardipine).",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Hypertension",
    },
    # --- HEART FAILURE (3) ---
    {
        "Question": "In patients with heart failure with reduced ejection fraction (HFrEF), what are the 'four pillars' of medical therapy?",
        "Answer": "ACEi/ARNI, beta-blockers, MRA (mineralocorticoid receptor antagonists), and SGLT2 inhibitors.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    {
        "Question": "What is the ejection fraction threshold that defines heart failure with preserved ejection fraction (HFpEF) according to ESC 2021 guidelines?",
        "Answer": "LVEF ≥ 50%, in the presence of relevant structural/functional cardiac abnormalities and elevated natriuretic peptides.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    {
        "Question": "What is the mechanism of action of sacubitril in the ARNI combination (sacubitril/valsartan)?",
        "Answer": "Sacubitril inhibits neprilysin, preventing degradation of natriuretic peptides (ANP, BNP), leading to vasodilation, natriuresis, and reduced cardiac remodelling.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Heart Failure",
    },
    # --- DYSLIPIDAEMIA / PREVENTION (3) ---
    {
        "Question": "What is the target LDL-C level for patients at very high cardiovascular risk according to the 2019 ESC/EAS guidelines?",
        "Answer": "< 1.4 mmol/L (55 mg/dL) and a reduction of ≥ 50% from baseline.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    {
        "Question": "What is the first-line pharmacological treatment for hypercholesterolaemia?",
        "Answer": "High-intensity statin therapy (e.g., atorvastatin 40–80 mg or rosuvastatin 20–40 mg daily).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    {
        "Question": "What is the mechanism of action of PCSK9 inhibitors, and when are they indicated?",
        "Answer": "PCSK9 inhibitors (e.g., evolocumab, alirocumab) block the PCSK9 protein, preventing LDL receptor degradation and thus increasing LDL clearance. They are indicated in very high-risk patients not achieving LDL targets on maximum tolerated statin ± ezetimibe.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Dyslipidaemia",
    },
    # --- ACUTE CORONARY SYNDROME (3) ---
    {
        "Question": "In the context of acute coronary syndrome, what does the 'TIMI score' predict?",
        "Answer": "The risk of death and ischemic events in patients with UA/NSTEMI or STEMI.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    {
        "Question": "What is the door-to-balloon time target for primary PCI in STEMI according to ESC guidelines?",
        "Answer": "< 60 minutes from first medical contact if the patient presents directly to a PCI-capable centre.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    {
        "Question": "What dual antiplatelet therapy (DAPT) regimen is recommended after primary PCI for STEMI?",
        "Answer": "Aspirin plus a potent P2Y12 inhibitor (ticagrelor or prasugrel preferred over clopidogrel) for 12 months, unless there is a high bleeding risk.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Acute Coronary Syndrome",
    },
    # --- ATRIAL FIBRILLATION / ARRHYTHMIAS (4) ---
    {
        "Question": "What is the first-line pharmacological treatment for rate control in atrial fibrillation?",
        "Answer": "Beta-blockers or non-dihydropyridine calcium channel blockers (diltiazem or verapamil).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "According to ESC guidelines, what is the recommended anticoagulation therapy for non-valvular atrial fibrillation in eligible patients?",
        "Answer": "Direct oral anticoagulants (DOACs) are preferred over vitamin K antagonists (VKAs).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "What is the CHA₂DS₂-VASc score used for, and at what threshold is anticoagulation recommended in males?",
        "Answer": "It estimates stroke risk in atrial fibrillation; anticoagulation is recommended at a score ≥ 2 in males (≥ 3 in females).",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    {
        "Question": "What ECG finding is pathognomonic of Wolff-Parkinson-White (WPW) syndrome?",
        "Answer": "A short PR interval (< 120 ms), a delta wave (slurred QRS upstroke), and a wide QRS complex.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Arrhythmias",
    },
    # --- VALVULAR HEART DISEASE (3) ---
    {
        "Question": "What are the classic auscultatory findings of severe aortic stenosis?",
        "Answer": "A harsh crescendo-decrescendo systolic ejection murmur at the right upper sternal border, radiating to the carotids, with a slow-rising and low-volume pulse (pulsus parvus et tardus).",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    {
        "Question": "According to ESC guidelines, what is the threshold for surgical aortic valve replacement (SAVR) or TAVI in symptomatic severe aortic stenosis?",
        "Answer": "AVA < 1.0 cm², mean gradient > 40 mmHg, or peak velocity > 4 m/s with symptoms (angina, syncope, or dyspnoea). Intervention is indicated (Class I) once symptoms are present.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    {
        "Question": "What is the underlying mechanism of functional (secondary) mitral regurgitation in dilated cardiomyopathy?",
        "Answer": "Left ventricular dilation causes papillary muscle displacement and annular dilation, leading to incomplete mitral leaflet coaptation without intrinsic leaflet pathology.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Valvular Heart Disease",
    },
    # --- CARDIOMYOPATHIES (3) ---
    {
        "Question": "What is the most common genetic cause of hypertrophic cardiomyopathy (HCM)?",
        "Answer": "Mutations in genes encoding sarcomeric proteins, most commonly MYH7 (beta-myosin heavy chain) and MYBPC3 (myosin-binding protein C), inherited in an autosomal dominant pattern.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    {
        "Question": "What physical manoeuvre increases the murmur of hypertrophic obstructive cardiomyopathy (HOCM), and why?",
        "Answer": "The Valsalva manoeuvre (strain phase) increases the murmur by reducing preload, which decreases LV volume and worsens dynamic outflow tract obstruction.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    {
        "Question": "What is the first-line treatment for acute myocarditis with haemodynamic compromise?",
        "Answer": "Supportive care including diuretics, ACE inhibitors, and beta-blockers as tolerated; mechanical circulatory support (IABP or VA-ECMO) in refractory cardiogenic shock. Immunosuppression may be considered in biopsy-proven giant cell or autoimmune myocarditis.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Cardiomyopathy",
    },
    # --- PERICARDIAL DISEASE (2) ---
    {
        "Question": "What is the first-line treatment for acute pericarditis?",
        "Answer": "NSAIDs (e.g., ibuprofen or aspirin) plus colchicine for 3 months to reduce recurrence risk.",
        "Difficulty": "Easy",
        "Category": "Cardiology",
        "Sub-topic": "Pericardial Disease",
    },
    {
        "Question": "What are Beck's triad findings in cardiac tamponade?",
        "Answer": "Hypotension, elevated jugular venous pressure (JVP), and muffled heart sounds.",
        "Difficulty": "Medium",
        "Category": "Cardiology",
        "Sub-topic": "Pericardial Disease",
    },
    # --- AORTIC DISEASE (1) ---
    {
        "Question": "According to ESC guidelines, what is the diameter threshold for elective surgical repair of an ascending aortic aneurysm in a non-syndromic patient?",
        "Answer": "≥ 55 mm in diameter; the threshold is lower (≥ 50 mm) in patients with bicuspid aortic valve or connective tissue disorders such as Marfan syndrome.",
        "Difficulty": "Hard",
        "Category": "Cardiology",
        "Sub-topic": "Aortic Disease",
    },
]

data = pd.DataFrame(medical_qa)

## Final Evaluation Loop
We re-run the evaluation and compare metrics.

In [3]:
results_list = []
instructions = "\nProvide a concise medical answer based on clinical guidelines."

model.eval()
for index, row in tqdm(data.iterrows(), total=len(data)):
    question = row["Question"]
    answer = row["Answer"]
    difficulty = row["Difficulty"]

    # Tokenise the prompt
    encoded = tokenizer.apply_chat_template(
        [{"role": "user", "content": question + instructions}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = encoded.to(device)
    # Build an explicit attention mask so generate() does not attend over
    # pad/eos tokens (the source of the "attention mask not set" warning and
    # the erratic continuations it causes).
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            do_sample=False,  # greedy — deterministic, fair comparison
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,  # mild penalty to avoid looping
        )

    generated_token_ids = outputs[0, input_ids.shape[-1] :]
    generated_text = tokenizer.decode(
        generated_token_ids, skip_special_tokens=True
    ).strip()

    # Perplexity on the reference answer
    reference_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_input_ids = torch.cat([input_ids, reference_ids], dim=1)
    labels = full_input_ids.clone()
    labels[:, : input_ids.shape[1]] = -100  # mask the prompt
    ref_attn_mask = torch.ones_like(full_input_ids)
    with torch.no_grad():
        loss = model(full_input_ids, attention_mask=ref_attn_mask, labels=labels).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated_text,
            "difficulty": difficulty,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
results_df.head()

100%|██████████| 25/25 [01:28<00:00,  3.52s/it]


,question,expected_answer,generated_answer,difficulty,perplexity
0,What is the recommended first-line treatment f...,"ACE inhibitors or ARBs, combined with a calciu...",Semesterly measures conflict which acceptable ...,Easy,31903.681641
1,What is the definition of Stage 1 hypertension...,Systolic blood pressure 130–139 mmHg or diasto...,Symptom and reach-of-Add blood pressure indica...,Medium,457.230164
2,What is the definition of hypertensive emergen...,Hypertensive emergency is severe hypertension ...,Of doctors who have been are they also breakin...,Hard,22393.783203
3,In patients with heart failure with reduced ej...,"ACEi/ARNI, beta-blockers, MRA (mineralocortico...",Backbench HFr3C equipped with acquired strengt...,Easy,13439.069336
4,What is the ejection fraction threshold that d...,"LVEF ≥ 50%, in the presence of relevant struct...",Ewed aloud and levels persuasive `ratios` (whi...,Medium,21229.544922


## Scoring Metrics

In [4]:
def calculate_keyword_score(generated, expected):
    expected_keywords = set(re.findall(r"\b\w{5,}\b", expected.lower()))
    if not expected_keywords:
        return 1.0
    generated_lower = generated.lower()
    found = sum(1 for kw in expected_keywords if kw in generated_lower)
    return found / len(expected_keywords)


similarity_model = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_semantic_similarity(generated, expected):
    emb1 = similarity_model.encode(generated, convert_to_tensor=True)
    emb2 = similarity_model.encode(expected,  convert_to_tensor=True)
    return util.pytorch_cos_sim(emb1, emb2).item()


JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type    = "nf4",
    bnb_4bit_use_double_quant = True,
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model     = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)


def ai_judge_score(question, generated, expected):
    judge_prompt = (
        "You are a medical board examiner.\n"
        "Rate the following generated answer on a scale from 1 to 10 "
        "based on its accuracy compared to the reference answer.\n\n"
        f"Question: {question}\n"
        f"Reference Answer: {expected}\n"
        f"Generated Answer: {generated}\n\n"
        "Rules:\n"
        "- 10: Perfectly accurate and comprehensive.\n"
        "- 7-9: Mostly accurate with minor omissions.\n"
        "- 4-6: Partially correct but missing key clinical details.\n"
        "- 1-3: Incorrect or dangerously misleading.\n\n"
        "Return ONLY the numeric score (e.g., '8')."
    )
    messages = [{"role": "user", "content": judge_prompt}]
    inputs   = judge_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(judge_model.device)
    attn_mask = torch.ones_like(inputs)   # ← fix: same pad-token ambiguity as main model
    with torch.no_grad():
        outputs    = judge_model.generate(
            inputs,
            attention_mask = attn_mask,
            max_new_tokens = 10,
            do_sample      = False,
        )
        score_text = judge_tokenizer.decode(
            outputs[0, inputs.shape[-1]:], skip_special_tokens=True
        ).strip()
    match = re.search(r"\d+", score_text)
    return int(match.group(0)) / 10.0 if match else 0.5


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
def calculate_keyword_score(generated, expected):
    expected_keywords = set(re.findall(r"\b\w{5,}\b", expected.lower()))
    if not expected_keywords:
        return 1.0
    generated_lower = generated.lower()
    found = sum(1 for kw in expected_keywords if kw in generated_lower)
    return found / len(expected_keywords)


similarity_model = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_semantic_similarity(generated, expected):
    emb1 = similarity_model.encode(generated, convert_to_tensor=True)
    emb2 = similarity_model.encode(expected,  convert_to_tensor=True)
    return util.pytorch_cos_sim(emb1, emb2).item()


JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type    = "nf4",
    bnb_4bit_use_double_quant = True,
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model     = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)


def ai_judge_score(question, generated, expected):
    judge_prompt = (
        "You are a medical board examiner.\n"
        "Rate the following generated answer on a scale from 1 to 10 "
        "based on its accuracy compared to the reference answer.\n\n"
        f"Question: {question}\n"
        f"Reference Answer: {expected}\n"
        f"Generated Answer: {generated}\n\n"
        "Rules:\n"
        "- 10: Perfectly accurate and comprehensive.\n"
        "- 7-9: Mostly accurate with minor omissions.\n"
        "- 4-6: Partially correct but missing key clinical details.\n"
        "- 1-3: Incorrect or dangerously misleading.\n\n"
        "Return ONLY the numeric score (e.g., '8')."
    )
    messages = [{"role": "user", "content": judge_prompt}]
    inputs   = judge_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(judge_model.device)
    attn_mask = torch.ones_like(inputs)   # ← fix: same pad-token ambiguity as main model
    with torch.no_grad():
        outputs    = judge_model.generate(
            inputs,
            attention_mask = attn_mask,
            max_new_tokens = 10,
            do_sample      = False,
        )
        score_text = judge_tokenizer.decode(
            outputs[0, inputs.shape[-1]:], skip_special_tokens=True
        ).strip()
    match = re.search(r"\d+", score_text)
    return int(match.group(0)) / 10.0 if match else 0.5


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
results_df.generated_answer.iloc[5]

"Pads no stand-alone measure suggest to enhance received disease risk (e.g., highlighting main weakness) all-pervasive rate than mga in CR normalised with ANCH-F DNA press plate and enhanced shown some relevant weight bearing or abuse potential side-effects {7 day embodiment under capacity FDPs' test)."